xarray > numpy > mk tau
https://scipy.github.io/devdocs/reference/generated/scipy.stats.kendalltau.html


In [2]:
# Packages
import os
import xarray as xr
import rioxarray as rxr
import numpy as np
from scipy.stats import kendalltau

In [3]:
# Folder with your TIFF files
files_input = "/home/gisuser/code/data/3_MovingWindow/mw_area"

# Extract year from filename like '1990_area_1km.tif'
def extracted_year(filename):
    base = os.path.basename(filename)
    year_str = base.split('_')[0] # Result: "1990"
    return int(year_str) # returns int

# List TIFF files sorted by year
file_paths = sorted(
    [os.path.join(files_input, f)
    for f in os.listdir(files_input)
    if f.lower().endswith(('.tif', '.tiff'))],
    key=extracted_year
)

In [4]:
# Load files as xarray DataArrays and squeeze band dimension
#arrays = [rxr.open_rasterio(fp, masked=True).squeeze() for fp in file_paths]
arrays = [rxr.open_rasterio(fp, masked=True, chunks={'x':1024, 'y':1024}).squeeze() for fp in file_paths]

In [5]:
# Stack along time dimension (coordinates default 0,1,...)
stacked = xr.concat(arrays, dim='time')

In [6]:
print(stacked['time'])

<xarray.DataArray 'time' (time: 31)> Size: 248B
array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
Coordinates:
    band         int64 8B 1
    spatial_ref  int64 8B 0
Dimensions without coordinates: time


In [15]:
# calulate the number of NaN in a given year to properly set minimum number of years with data present to be counted towards mk tau

# Assume 'stacked' is your xarray.DataArray with dims ('time', 'y', 'x')
total_years = stacked.sizes['time']

# Count number of NaNs per pixel across time
nan_count = stacked.isnull().sum(dim='time')

# Mask pixels where all values are NaN (nan_count == total_years)
valid_pixels_mask = nan_count < total_years

# Calculate valid years count for pixels that have any data
valid_count = total_years - nan_count

# Apply mask to exclude pixels with all NaNs for min and mean calculations
valid_count_masked = valid_count.where(valid_pixels_mask)

# Compute statistics ignoring fully missing pixels
max_missing = nan_count.where(valid_pixels_mask).max().compute().item()
min_valid = valid_count_masked.min().compute().item()
mean_valid = valid_count_masked.mean().compute().item()

print(f"Maximum number of missing years (excluding fully missing pixels): {max_missing}")
print(f"Minimum number of represented years (excluding fully missing pixels): {min_valid}")
print(f"Mean number of represented years per pixel (excluding fully missing pixels): {mean_valid:.2f}")

Maximum number of missing years (excluding fully missing pixels): 0.0
Minimum number of represented years (excluding fully missing pixels): 31.0
Mean number of represented years per pixel (excluding fully missing pixels): 31.00


In [5]:
# Define Mann-Kendall Tau calculation using scipy's kendalltau
# uses numpy (only works in numpy)
# If enough valid data exists (≥3 time points), the output pixel gets its trend tau value.
# If not enough data points remain, the pixel in the output image will be NaN — indicating no reliable trend could be calculated for that pixel.
# Other pixels with sufficient data are unaffected; their tau values are calculated independently.
# tau-b is used by default in scipy's kendalltau function
def mk_tau(arr):
    arr = arr[~np.isnan(arr)]  # Filter NaNs
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

# converts to numpy and:
# Applies pixel-wise along time axis (axis=0)
tau_vals = np.apply_along_axis(mk_tau, axis=0, arr=stacked.values)

# Save output Tau raster using metadata from first file
# first_raster = arrays[0]
# transform = first_raster.rio.transform()
# height, width = tau_vals.shape

# out_meta = {
#     "driver": "GTiff",
#     "height": height,
#     "width": width,
#     "count": 1,
#     "dtype": "float32",
#     "crs": first_raster.rio.crs,
#     "transform": transform
# }

# output_path = "mann_kendall_tau_result.tif"
# with rasterio.open(output_path, "w", **out_meta) as dst:
#     dst.write(tau_vals.astype(np.float32), 1)

# print(f"Mann-Kendall Tau image saved to {output_path}")


: 

: 

: 

In [ ]:
# Save output Tau raster using metadata from first file
first_raster = arrays[0]
transform = first_raster.rio.transform()
height, width = tau_vals.shape

out_meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": first_raster.rio.crs,
    "transform": transform
}

output_path = "mann_kendall_tau_result.tif"
with rasterio.open(output_path, "w", **out_meta) as dst:
    dst.write(tau_vals.astype(np.float32), 1)

print(f"Mann-Kendall Tau image saved to {output_path}")

In [ ]:
# tile the data

def split_tiles(raster_data, tile_row, tile_col):
    num_rows, num_cols = raster_data.sizes['y'], raster_data.sizes['x']
    tile_height = num_rows // tile_row
    tile_width = num_cols // tile_col
    
    tiles = []
    for i in range(tile_row):
        for j in range(tile_col):
            row_start = i * tile_height
            row_end = (i + 1) * tile_height if i != tile_row - 1 else num_rows
            col_start = j * tile_width
            col_end = (j + 1) * tile_width if j != tile_col - 1 else num_cols
            
            tile_data = raster_data.isel(y=slice(row_start, row_end), x=slice(col_start, col_end))
            tiles.append(tile_data)
    
    return tiles

# Aiyin's code
# def split_tiles(self,tile_row,tile_col):
#         raster_data = self.input_
#         # Get the number of rows (y dimension) and columns (x dimension)
#         num_rows, num_cols = raster_data.sizes['y'], raster_data.sizes['x']
        
#         # Define the size of each tile (5x5 grid)
#         tile_height = num_rows // tile_row  # Rows per tile
#         tile_width = num_cols // tile_col   # Columns per tile
        
#         tile_idx = 0
        
#         tile_datasets = []
#         nodata_val = self.nodata_val
#         # Loop through the 5x5 grid of tiles
#         for i in range(tile_row):
#             for j in range(tile_col):
#                 print(i,j)
#                 # Calculate the index range for this tile
#                 row_start = i * tile_height
#                 row_end = (i + 1) * tile_height if i != tile_row - 1 else num_rows
#                 col_start = j * tile_width
#                 col_end = (j + 1) * tile_width if j != tile_col - 1 else num_cols
                
#                 # Slice the data for this tile based on indices
#                 tile_data = raster_data.isel(y=slice(row_start, row_end), x=slice(col_start, col_end))
                
#                 valid_data = tile_data != nodata_val 
#                 #print(tile_data)
#                 if(valid_data.any().compute()):
#                     tile_datasets.append(tile_data) 
        
#         return tile_datasets

In [5]:
# with dask
import dask.array as da
# Initialize Dask cluster
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=4, threads_per_worker=1, memory_limit='8GB')
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

Dashboard: http://127.0.0.1:8787/status


In [ ]:
def mk_tau(arr):
    arr = arr[~np.isnan(arr)]
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

def mk_tau_block(block):
    return np.apply_along_axis(mk_tau, axis=0, arr=block)

# Use da.map_blocks
tau_result = da.map_blocks(
    mk_tau_block,
    stacked.data,
    drop_axis=0,
    dtype=np.float32
)

# Compute result
tau_vals = tau_result.compute()

# Save result
first_raster = arrays[0]
out_meta = {
    "driver": "GTiff",
    "height": tau_vals.shape[0],
    "width": tau_vals.shape[1],
    "count": 1,
    "dtype": "float32",
    "crs": first_raster.rio.crs,
    "transform": first_raster.rio.transform()
}

with rasterio.open("mann_kendall_tau_result.tif", "w", **out_meta) as dst:
    dst.write(tau_vals, 1)

/opt/conda/envs/base_gis/lib/python3.12/site-packages/distributed/client.py:3361: UserWarning: Sending large graph of size 52.01 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:


def mk_tau(arr):
    arr = arr[~np.isnan(arr)]
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

def split_tiles(raster_data, tile_row, tile_col):
    num_rows, num_cols = raster_data.sizes['y'], raster_data.sizes['x']
    tile_height = num_rows // tile_row
    tile_width = num_cols // tile_col
    
    tiles = []
    for i in range(tile_row):
        for j in range(tile_col):
            row_start = i * tile_height
            row_end = (i + 1) * tile_height if i != tile_row - 1 else num_rows
            col_start = j * tile_width
            col_end = (j + 1) * tile_width if j != tile_col - 1 else num_cols
            
            tile_data = raster_data.isel(y=slice(row_start, row_end), x=slice(col_start, col_end))
            tiles.append(tile_data)
    
    return tiles

def mk_tau_block(block):
    return np.apply_along_axis(mk_tau, axis=0, arr=block)

# Split data into tiles
tiles = split_tiles(stacked, 5, 5)

# Process each tile
processed_tiles = []
for tile in tiles:
    tau_result = da.map_blocks(
        mk_tau_block,
        tile.data,
        drop_axis=0,
        dtype=np.float32
    )
    
    tau_tile = xr.DataArray(
        tau_result,
        coords={'y': tile.y, 'x': tile.x},
        dims=['y', 'x']
    )
    processed_tiles.append(tau_tile)

# Combine tiles
tau_combined = xr.combine_by_coords(processed_tiles)

# Save with xarray (simpler)
tau_combined.rio.write_crs(arrays[0].rio.crs, inplace=True)
tau_combined.rio.write_transform(arrays[0].rio.transform(), inplace=True)
tau_combined.rio.to_raster("mann_kendall_tau_result.tif")

print("Mann-Kendall Tau image saved")


/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 81
  result = blockwise(
/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 81
  result = blockwise(
/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 81
  result = blockwise(
/opt/conda/envs/base_gis/lib/python3.12/site-packages/dask/array/core.py:4894: PerformanceWarning: Increasing number of chunks by factor of 75
  result = blockwise(
/opt/conda

2025-09-17 22:27:36,410 - distributed.worker - ERROR - 
Traceback (most recent call last):
  File "/opt/conda/envs/base_gis/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/base_gis/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/base_gis/lib/python3.12/asyncio/base_events.py", line 678, in run_until_complete
    self.run_forever()
  File "/opt/conda/envs/base_gis/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/opt/conda/envs/base_gis/lib/python3.12/asyncio/base_events.py", line 1961, in _run_once
    event_list = self._selector.select(timeout)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/base_gis/lib/python3.12/selectors.py", line 468, in select
    fd_event_list = self._selector.poll(timeout, max_ev)
       

In [ ]:
#stack > tiled
#tau
# with dask
import dask.array as da
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=4, threads_per_worker=1, memory_limit='8GB')
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

def mk_tau(arr):
    arr = arr[~np.isnan(arr)]
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

def mk_tau_block(block):
    """Apply mk_tau to each pixel in a block"""
    return np.apply_along_axis(mk_tau, axis=0, arr=block)

# Rechunk stacked data for tiling
stacked_rechunked = stacked.chunk({'time': -1, 'y': 512, 'x': 512})

# Apply mk_tau using map_blocks
tau_result = da.map_blocks(
    mk_tau_block,
    stacked_rechunked.data,
    drop_axis=0,
    dtype=np.float32
)

# Convert to xarray
tau_xr = xr.DataArray(
    tau_result,
    dims=['y', 'x'],
    coords={'y': stacked.y, 'x': stacked.x}
)

# Compute and save
tau_computed = tau_xr.compute()
tau_computed.rio.write_crs(arrays[0].rio.crs, inplace=True)
tau_computed.rio.write_transform(arrays[0].rio.transform(), inplace=True)
tau_computed.rio.to_raster("mann_kendall_tau_result.tif")

print("Mann-Kendall Tau image saved")
client.close()


Dashboard: http://127.0.0.1:8787/status


/opt/conda/envs/base_gis/lib/python3.12/site-packages/distributed/client.py:3361: UserWarning: Sending large graph of size 60.41 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2025-09-17 23:10:13,789 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 4.47 GiB -- Worker memory limit: 7.45 GiB
2025-09-17 23:10:31,835 - tornado.application - ERROR - Uncaught exception GET /status/ws (172.17.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='172.

2025-09-17 23:34:11,444 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('open_rasterio-getitem-fe0d54689e49c3674045c7543563dabf', 91, 36))" coro=<Worker.execute() done, defined at /opt/conda/envs/base_gis/lib/python3.12/site-packages/distributed/worker_state_machine.py:3609>> ended with CancelledError
2025-09-17 23:34:11,446 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('open_rasterio-getitem-a1d9c02343755b130644a41e1c451ec5', 75, 1))" coro=<Worker.execute() done, defined at /opt/conda/envs/base_gis/lib/python3.12/site-packages/distributed/worker_state_machine.py:3609>> ended with CancelledError
2025-09-17 23:34:11,714 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:39859' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('concatenate-2c8dfe549c2f467ed4188d9644bc74a8', 15, 91, 57), ('concatenate-2c8dfe549c2f467ed4188d964

In [ ]:
# give me another option that tiles the data 5x7 so 35 total tiles and dont use dask
def mk_tau(arr):
    arr = arr[~np.isnan(arr)]
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

# Tile data into 5x7 grid (35 tiles)
rows, cols = stacked.sizes['y'], stacked.sizes['x']
tile_rows, tile_cols = 5, 7
tile_height = rows // tile_rows
tile_width = cols // tile_cols

# Process each tile
tau_tiles = []
for i in range(tile_rows):
    for j in range(tile_cols):
        # Calculate tile boundaries
        y_start = i * tile_height
        y_end = (i + 1) * tile_height if i != tile_rows - 1 else rows
        x_start = j * tile_width
        x_end = (j + 1) * tile_width if j != tile_cols - 1 else cols
        
        # Extract tile
        tile = stacked.isel(y=slice(y_start, y_end), x=slice(x_start, x_end))
        
        # Calculate tau for tile
        tau_tile = np.apply_along_axis(mk_tau, axis=0, arr=tile.values)
        
        # Store with coordinates
        tau_xr = xr.DataArray(
            tau_tile,
            dims=['y', 'x'],
            coords={'y': tile.y, 'x': tile.x}
        )
        tau_tiles.append(tau_xr)
        
        print(f"Processed tile {i*tile_cols + j + 1}/35")

# Combine all tiles
tau_combined = xr.combine_by_coords(tau_tiles)

# Save result
tau_combined.rio.write_crs(arrays[0].rio.crs, inplace=True)
tau_combined.rio.write_transform(arrays[0].rio.transform(), inplace=True)
tau_combined.rio.to_raster("mann_kendall_tau_result.tif")

print("Mann-Kendall Tau image saved")


In [6]:

def mk_tau(arr):
    arr = arr[~np.isnan(arr)]
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

def split_tiles(raster_data, tile_row, tile_col):
    num_rows, num_cols = raster_data.sizes['y'], raster_data.sizes['x']
    tile_height = num_rows // tile_row
    tile_width = num_cols // tile_col
    
    tiles = []
    for i in range(tile_row):
        for j in range(tile_col):
            row_start = i * tile_height
            row_end = (i + 1) * tile_height if i != tile_row - 1 else num_rows
            col_start = j * tile_width
            col_end = (j + 1) * tile_width if j != tile_col - 1 else num_cols
            
            tile_data = raster_data.isel(y=slice(row_start, row_end), x=slice(col_start, col_end))
            tiles.append((tile_data, row_start, row_end, col_start, col_end))
    
    return tiles

# Split into tiles
tiles = split_tiles(stacked, 8, 8)

# Initialize output array
height, width = stacked.shape[1], stacked.shape[2]
tau_vals = np.full((height, width), np.nan, dtype=np.float32)

# Process each tile
for i, (tile_data, row_start, row_end, col_start, col_end) in enumerate(tiles):
    print(f"Processing tile {i+1}/{len(tiles)}")
    
    # Convert to numpy and process
    tile_array = tile_data.compute().values
    tile_tau = np.apply_along_axis(mk_tau, axis=0, arr=tile_array)
    
    # Place in output array
    tau_vals[row_start:row_end, col_start:col_end] = tile_tau

# Save result
first_raster = arrays[0]
out_meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": first_raster.rio.crs,
    "transform": first_raster.rio.transform()
}

with rasterio.open("area_TS_MKtau.tif", "w", **out_meta) as dst:
    dst.write(tau_vals, 1)


Processing tile 1/64


KeyboardInterrupt: 